# L4: Contextual Retrieval and Filtering

Vector similarity alone isn't enough. When your device stores hundreds of memories throughout the week, you need to filter by **time**, **location**, and **type** to get relevant results fast.

In this lesson, you'll learn to:
- Use payload filtering to narrow search results
- Build time-windowed queries ("what happened this afternoon?")
- Filter by location and category
- Combine multiple filters with `must` (AND) and `should` (OR)
- Benchmark filtered vs. unfiltered query latency at scale

## Setup

In [ ]:
!pip install qdrant-edge-py qai-hub "qai-hub-models[nomic_embed_text]" torch transformers numpy

In [ ]:
import os
import sys
sys.path.append("..")

import torch
import numpy as np
import time
from pathlib import Path
from transformers import AutoTokenizer
from qai_hub_models.models.nomic_embed_text import Model as NomicEmbedText
from qdrant_edge import (
    EdgeShard, EdgeConfig, VectorDataConfig, Distance,
    Point, UpdateOperation, Query, QueryRequest,
)

# Load embedding model and tokenizer
text_model = NomicEmbedText.from_pretrained()
tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5")

EMBEDDING_DIM = 512
MAX_SEQ_LEN = 512

def embed(texts, prefix="search_document: "):
    """Generate embeddings using the nomic model."""
    prefixed = [prefix + t for t in texts]
    encoded = tokenizer(
        prefixed, padding="max_length", truncation=True,
        max_length=MAX_SEQ_LEN, return_tensors="pt",
    )
    with torch.no_grad():
        embeddings = text_model(encoded["input_ids"], encoded["attention_mask"])
    return embeddings

print(f"Embedding model loaded: nomic_embed_text ({EMBEDDING_DIM}d)")

### Compile for AI Hub (Optional)

You can compile this model for on-device deployment, just like in L3. Here we show the compile step for reference.

In [ ]:
import qai_hub
from utils import get_ai_hub_api_token, get_random_device

ai_hub_api_token = get_ai_hub_api_token()
!qai-hub configure --api_token $ai_hub_api_token

device_name = get_random_device()
device = qai_hub.Device(device_name)

# Trace and compile
example_ids = torch.randint(0, tokenizer.vocab_size, (1, MAX_SEQ_LEN))
example_mask = torch.ones(1, MAX_SEQ_LEN, dtype=torch.long)
traced_model = torch.jit.trace(text_model, (example_ids, example_mask))

compile_job = qai_hub.submit_compile_job(
    model=traced_model,
    input_specs={"input_ids": (1, MAX_SEQ_LEN), "attention_mask": (1, MAX_SEQ_LEN)},
    device=device,
)
target_model = compile_job.get_target_model()
print(f"Model compiled for {device_name}")

## 1. Build a Realistic Memory Store

Let's populate an EdgeShard with a full week of memories from multiple devices and locations. A real device accumulates hundreds of entries across days.

In [ ]:
SHARD_DIR = "./context_shard"
Path(SHARD_DIR).mkdir(parents=True, exist_ok=True)

VECTOR_NAME = "memory"

config = EdgeConfig(
    vector_data={
        VECTOR_NAME: VectorDataConfig(
            size=EMBEDDING_DIM,
            distance=Distance.Cosine,
        )
    }
)

shard = EdgeShard(SHARD_DIR, config)

# A week of memories across multiple devices and locations
# Each day has a mix of work, personal, and routine activities
memory_templates = {
    "meeting": [
        "Morning standup: discussed sprint velocity and blockers",
        "Design review for the new API endpoints",
        "1:1 with manager about project priorities",
        "All-hands meeting: company Q3 results presented",
        "Sprint planning session for the sync module",
        "Architecture review: database schema changes",
        "Team retro: discussed process improvements",
        "Cross-team sync on the mobile release timeline",
        "Demo day: showed the edge deployment prototype",
        "Security review for the authentication flow",
    ],
    "work": [
        "Reviewed pull request for the caching layer",
        "Fixed a race condition in the sync module",
        "Pair programming on the search optimization",
        "Code review: found a memory leak in the embeddings pipeline",
        "Deployed the new API version to staging",
        "Wrote unit tests for the filtering logic",
        "Profiled query latency on the test dataset",
        "Debugged the shard persistence issue on Android",
        "Refactored the embedding batch processor",
        "Updated the CI pipeline configuration",
    ],
    "food": [
        "Lunch at the taco place, tried the new fish tacos",
        "Morning coffee at the corner cafe, got an oat milk latte",
        "Dinner: cooked pasta with roasted vegetables",
        "Quick lunch: grabbed a sandwich from the deli",
        "Tried the new ramen spot downtown, excellent broth",
        "Breakfast: yogurt and granola at home",
        "Team lunch at the Thai restaurant",
        "Afternoon snack: apple and peanut butter",
        "Grilled salmon with asparagus for dinner",
        "Birthday cake for Maria in the break room",
    ],
    "social": [
        "Coffee break with Sarah, talked about weekend hiking",
        "Bumped into Alex at the elevator, chatted about the conference",
        "Video call with college friends planning a reunion trip",
        "Happy hour with the team after the release",
        "Phone call with mom about Thanksgiving plans",
        "Walked with John to get lunch, discussed the new project",
        "Board game night at Dave's apartment",
        "Caught up with the intern about their experience",
        "Slack thread debating the best IDE setup",
        "Group chat planning the team offsite",
    ],
    "exercise": [
        "Morning jog around the lake, spotted a heron",
        "Lunchtime yoga class at the office gym",
        "Evening bike ride through the park, 8 miles",
        "Quick 20-minute HIIT workout before work",
        "Walked 10,000 steps exploring the neighborhood",
        "Swimming laps at the community pool",
        "Stretching routine after sitting all day",
        "Took the stairs instead of elevator, 12 floors",
    ],
    "learning": [
        "Read about vector database indexing strategies",
        "Watched a talk on transformer architecture optimizations",
        "Studied the Qdrant Edge documentation for filtering",
        "Listened to a podcast about edge computing trends",
        "Read the HNSW paper for better understanding of ANN search",
        "Tutorial on Snapdragon NPU programming",
        "Skimmed the new CLIP paper on zero-shot classification",
        "Reviewed AI Hub model zoo documentation",
    ],
    "errand": [
        "Grocery run: picked up salmon and vegetables",
        "Dropped off dry cleaning on the way to work",
        "Picked up a package from the post office",
        "Stopped at the pharmacy for allergy medication",
        "Got gas and a car wash",
        "Returned shoes that didn't fit at the mall",
        "Picked up prescription glasses from the optometrist",
        "Hardware store: bought screws for the shelf project",
    ],
}

locations_by_category = {
    "meeting": "office", "work": "office", "food": "restaurant",
    "social": "various", "exercise": "outdoors", "learning": "home",
    "errand": "store",
}

devices_by_category = {
    "meeting": "glasses", "work": "glasses", "food": "phone",
    "social": "phone", "exercise": "watch", "learning": "phone",
    "errand": "phone",
}

# Generate a week of memories (Mon-Fri)
np.random.seed(42)
all_memories = []
day_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

for day_idx, day_name in enumerate(day_names):
    # Each day has 15-20 memories across categories
    day_categories = list(memory_templates.keys())
    for cat in day_categories:
        templates = memory_templates[cat]
        # Pick 2-3 random memories from each category per day
        n_picks = np.random.randint(2, 4)
        picked = np.random.choice(len(templates), size=min(n_picks, len(templates)), replace=False)
        for idx in picked:
            hour = np.random.randint(7, 22)
            all_memories.append({
                "text": f"{templates[idx]}",
                "location": locations_by_category[cat],
                "device": devices_by_category[cat],
                "category": cat,
                "day": day_name,
                "day_idx": day_idx,
                "hour": hour,
            })

# Shuffle to simulate natural order
np.random.shuffle(all_memories)

# Embed all memories
texts = [m["text"] for m in all_memories]
print(f"Embedding {len(texts)} memories...")
# Batch embed to avoid memory issues
batch_size = 16
all_embeddings = []
for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    all_embeddings.append(embed(batch))
embeddings = torch.cat(all_embeddings, dim=0)

week_start = time.time() - 5 * 24 * 3600  # Monday

points = []
for i, (m, emb) in enumerate(zip(all_memories, embeddings)):
    ts = week_start + m["day_idx"] * 86400 + m["hour"] * 3600
    points.append(Point(
        id=i,
        vector={VECTOR_NAME: emb.tolist()},
        payload={
            "text": m["text"],
            "location": m["location"],
            "device": m["device"],
            "category": m["category"],
            "day": m["day"],
            "timestamp": ts,
            "hour": m["hour"],
        }
    ))

shard.update(UpdateOperation.upsert_points(points))

# Print distribution
from collections import Counter
cat_counts = Counter(m["category"] for m in all_memories)
day_counts = Counter(m["day"] for m in all_memories)

print(f"\nStored {len(points)} memories across the week")
print(f"By day:      {dict(sorted(day_counts.items(), key=lambda x: day_names.index(x[0])))}")
print(f"By category: {dict(cat_counts)}")

## 2. Basic Payload Filtering

Qdrant Edge supports filtering results by payload fields. You can match exact values, ranges, and combine conditions.

The filtering uses the same format as the Qdrant server. Filters are passed via the `filter` parameter in `QueryRequest`.

In [ ]:
def search_with_filter(query_text, payload_filter=None, limit=5):
    query_emb = embed([query_text], prefix="search_query: ")

    request_kwargs = {
        "query": Query.Nearest(query_emb[0].tolist(), using=VECTOR_NAME),
        "limit": limit,
        "with_vector": False,
        "with_payload": True,
    }

    if payload_filter:
        request_kwargs["filter"] = payload_filter

    results = shard.query(QueryRequest(**request_kwargs))
    return results

def print_results(results):
    for r in results:
        p = r.payload
        print(f"  [{r.score:.3f}] [{p['category']}] [{p['location']}] {p['text']}")

### Filter by Location

Find memories from the office only.

In [ ]:
print("Query: 'technical discussion' filtered to office")
results = search_with_filter(
    "technical discussion",
    payload_filter={"must": [{"key": "location", "match": {"value": "office"}}]}
)
print_results(results)

### Filter by Category

In [ ]:
print("Query: 'what happened today' filtered to meetings only")
results = search_with_filter(
    "what happened today",
    payload_filter={"must": [{"key": "category", "match": {"value": "meeting"}}]}
)
print_results(results)

### Filter by Device

In [ ]:
print("Query: 'what did I see' filtered to glasses only")
results = search_with_filter(
    "what did I see",
    payload_filter={"must": [{"key": "device", "match": {"value": "glasses"}}]}
)
print_results(results)

## 3. Time-Windowed Queries

Time is critical for on-device memory. "What did I do Wednesday afternoon?" needs a time range filter.

In [ ]:
# Wednesday afternoon: day_idx=2, hours 12-17
wed_afternoon_start = week_start + 2 * 86400 + 12 * 3600
wed_afternoon_end = week_start + 2 * 86400 + 17 * 3600

print("Query: 'what happened' filtered to Wednesday afternoon (12pm-5pm)")
results = search_with_filter(
    "what happened",
    payload_filter={
        "must": [
            {"key": "timestamp", "range": {"gte": wed_afternoon_start, "lte": wed_afternoon_end}}
        ]
    }
)
print_results(results)

print()

# Last 24 hours
last_24h = time.time() - 86400
print("Query: 'food and meals' filtered to last 24 hours")
results = search_with_filter(
    "food and meals",
    payload_filter={
        "must": [
            {"key": "timestamp", "range": {"gte": last_24h}}
        ]
    }
)
print_results(results)

## 4. Combined Filters

Combine multiple conditions. Each entry in the `must` list is ANDed together.

In [ ]:
# Work meetings at the office on Monday and Tuesday only
tue_end = week_start + 2 * 86400

print("Query: 'discussion' filtered to office + meeting + Mon/Tue")
results = search_with_filter(
    "discussion",
    payload_filter={
        "must": [
            {"key": "location", "match": {"value": "office"}},
            {"key": "category", "match": {"value": "meeting"}},
            {"key": "timestamp", "range": {"lte": tue_end}},
        ]
    }
)
print_results(results)

## 5. "Should" Filters (OR Logic)

Use `should` for OR conditions. This matches points that satisfy at least one condition.

In [ ]:
print("Query: 'healthy activities' (food OR exercise)")
results = search_with_filter(
    "healthy activities",
    payload_filter={
        "should": [
            {"key": "category", "match": {"value": "food"}},
            {"key": "category", "match": {"value": "exercise"}},
        ]
    }
)
print_results(results)

## 6. Latency Benchmark: Filtered vs. Unfiltered

In [ ]:
query_emb = embed(["work meeting"], prefix="search_query: ")[0].tolist()
n_runs = 200

# Unfiltered
unfiltered_times = []
for _ in range(n_runs):
    start = time.perf_counter()
    shard.query(QueryRequest(
        query=Query.Nearest(query_emb, using=VECTOR_NAME),
        limit=5, with_vector=False, with_payload=True,
    ))
    unfiltered_times.append((time.perf_counter() - start) * 1000)

# Filtered
filtered_times = []
for _ in range(n_runs):
    start = time.perf_counter()
    shard.query(QueryRequest(
        query=Query.Nearest(query_emb, using=VECTOR_NAME),
        limit=5, with_vector=False, with_payload=True,
        filter={"must": [
            {"key": "location", "match": {"value": "office"}},
            {"key": "category", "match": {"value": "meeting"}},
        ]}
    ))
    filtered_times.append((time.perf_counter() - start) * 1000)

u = np.array(unfiltered_times)
f = np.array(filtered_times)

print(f"Unfiltered: mean={u.mean():.2f}ms, P95={np.percentile(u, 95):.2f}ms")
print(f"Filtered:   mean={f.mean():.2f}ms, P95={np.percentile(f, 95):.2f}ms")

## 7. Cleanup

In [ ]:
shard.close()

import shutil
shutil.rmtree(SHARD_DIR, ignore_errors=True)
print("Cleaned up")

## Summary

In this lesson you learned how to:
- Filter queries by payload fields using `must` (AND) and `should` (OR) conditions
- Build time-windowed queries using `range` filters on timestamps
- Combine location, category, device, and time filters
- Benchmark query latency with and without filters

In the next lesson, you'll learn how to orchestrate memory across multiple devices with cascade queries and cloud sync.